# Lung Cancer CT Scan Classification

**Task:** 6-class classification of lung CT scans
- Benign
- Malignant
- Adenocarcinoma
- Large Cell Carcinoma
- Normal
- Squamous Cell Carcinoma

**Model:** EfficientNet-B0

**Dataset:** [CT Scan Images for Lung Cancer (Kaggle)](https://www.kaggle.com/datasets/dishantrathi20/ct-scan-images-for-lung-cancer)

In [ ]:
# ── Setup ──────────────────────────────────────────────────
import sys
sys.path.insert(0, '../..')  # Add project root

import torch
from shared.config import DEVICE, BATCH_SIZE, EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE
from shared.models import create_model
from shared.pipelines.dataset import create_dataloaders
from shared.pipelines.train import train_model

print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Load Configuration ─────────────────────────────────────
from config import PROJECT_ID, MODEL_NAME, CLASSES, NUM_CLASSES, IMG_SIZE, CLASS_COLORS
from shared.config import get_trained_model_path

print(f"Project: {PROJECT_ID}")
print(f"Model: {MODEL_NAME}")
print(f"Classes ({NUM_CLASSES}): {CLASSES}")
print(f"Image Size: {IMG_SIZE}")

In [ ]:
# ── Load Dataset ───────────────────────────────────────────
data_root = "data"
train_loader, val_loader, test_loader, dataset_classes = create_dataloaders(
    data_root=data_root,
    class_names=CLASSES,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=2,
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

In [ ]:
# ── Visualise Sample Images ────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

data_iter = iter(train_loader)
images, labels = next(data_iter)

# Denormalize
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = img * std + mean
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(CLASSES[labels[i]], fontsize=9, fontweight='bold')
        ax.axis('off')
plt.suptitle('Lung CT Scan Training Samples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Create Model ───────────────────────────────────────────
model = create_model(MODEL_NAME, num_classes=len(CLASSES))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {MODEL_NAME}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# ── Train Model ────────────────────────────────────────────
checkpoint_path = str(get_trained_model_path(PROJECT_ID))
print(f"Checkpoint will be saved to: {checkpoint_path}")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_path=checkpoint_path,
)

print("\n✓ Training complete!")

In [ ]:
# ── Evaluate on Test Set ───────────────────────────────────
from shared.utils.metrics import compute_metrics
from shared.utils.visualization import plot_confusion_matrix
from IPython.display import display
import base64

# Load best model
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

metrics = compute_metrics(all_labels, all_preds, all_probs)

print(f"Test Accuracy:  {metrics['accuracy']:.4f}")
print(f"Test Precision: {metrics['precision']:.4f}")
print(f"Test Recall:    {metrics['recall']:.4f}")
print(f"Test F1-Score:  {metrics['f1_score']:.4f}")
if metrics.get('roc_auc'):
    print(f"Test ROC-AUC:   {metrics['roc_auc']:.4f}")

# Display confusion matrix
cm_b64 = plot_confusion_matrix(metrics['confusion_matrix'], CLASSES)
from IPython.display import Image as IPImage
display(IPImage(base64.b64decode(cm_b64)))

In [ ]:
# ── Training History Plot ──────────────────────────────────
import matplotlib.pyplot as plt

train_acc = [m['accuracy'] for m in history['train']]
val_acc = [m['accuracy'] for m in history['val']]
train_loss = [m['loss'] for m in history['train']]
val_loss = [m['loss'] for m in history['val']]

best_epoch = val_acc.index(max(val_acc)) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_acc, label='Train Acc', linewidth=2)
ax1.plot(val_acc, label='Val Acc', linewidth=2)
ax1.axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5, label=f'Best epoch {best_epoch}')
ax1.set_title('Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_loss, label='Train Loss', linewidth=2)
ax2.plot(val_loss, label='Val Loss', linewidth=2)
ax2.axvline(best_epoch - 1, color='green', linestyle='--', alpha=0.5)
ax2.set_title('Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Lung Cancer — {MODEL_NAME} Training History', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Best Validation Accuracy: {max(val_acc):.4f} (epoch {best_epoch})")

In [ ]:
# ── Visualise Predictions with XAI ─────────────────────────
from shared.explainers.xai_factory import run_all_explainers
from shared.pipelines.transforms import get_inference_transform
from PIL import Image
import os
import random

# Pick a random test image
test_dir = "data/test"
class_name = random.choice(CLASSES)
cls_dir = os.path.join(test_dir, class_name)
if os.path.isdir(cls_dir):
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if files:
        img_path = os.path.join(cls_dir, random.choice(files))
        print(f"True class: {class_name}")
        print(f"Image: {img_path}")

        pil_image = Image.open(img_path).convert('RGB')
        transform = get_inference_transform(IMG_SIZE)
        input_tensor = transform(pil_image).unsqueeze(0).to(DEVICE)

        result = run_all_explainers(
            model=model,
            image_tensor=input_tensor,
            class_names=CLASSES,
            device=DEVICE,
        )

        pred = result['predictions']
        print(f"Prediction: {pred['predicted_class']}")
        print(f"Confidence: {pred['confidence']:.2%}")

        # Show XAI visualisations
        from IPython.display import display, HTML
        html = '<div style="display:grid; grid-template-columns:repeat(2,1fr); gap:10px;">'
        for key, exp in result['explanations'].items():
            if 'overlay_base64' in exp:
                html += f'''
                    <div style="text-align:center; border:1px solid #ddd; border-radius:8px; padding:10px;">
                        <h4>{exp['label']}</h4>
                        <img src="data:image/png;base64,{exp['overlay_base64']}" style="width:100%; border-radius:4px;"/>
                        <p style="font-size:0.8rem; color:#666;">{exp['description']}</p>
                    </div>'''
        html += '</div>'
        display(HTML(html))

In [ ]:
# ── Per-Class Metrics ──────────────────────────────────────
from sklearn.metrics import classification_report

print("Classification Report:\n")
report = classification_report(
    all_labels, all_preds,
    target_names=CLASSES,
    digits=4
)
print(report)

---
**Next Steps:**
1. Model weights saved to `trained_models/11_lung_cancer_best.pth`
2. Run `python run.py` from project root to start the web interface
3. Upload a lung CT scan and see predictions with XAI heatmaps (Grad-CAM, Saliency, etc.)